# Iraq Flaring Watch — VIIRS "last detection" builder (FIRMS API)

This notebook matches each mapped gas flare against NASA FIRMS **VIIRS S-NPP** thermal detections
and records, for every flare, the **date it was last seen burning** and how many times it was
detected in the window.

It does **not** label a flare "active" or "inactive" — it only reports the verifiable fact
(last satellite detection date), which the website then shows in each flare's popup.

**This version pulls the detections straight from the NASA FIRMS Area API** using your MAP_KEY,
so there is nothing to download or upload. You just paste your key once and run all cells.

## What you do each quarter
1. Run all cells (Runtime → Run all).
2. When prompted the first time, paste your FIRMS **MAP_KEY** (it gets saved to a Colab Secret
   named `FIRMS_MAP_KEY`, so future runs don't ask again).
3. Download the generated `flare_viirs_status.json` and commit it to the repo root on GitHub.

The flare list is read live from your repo's `query` file, so flare IDs always match the website.

### A note on the two VIIRS products
- `VIIRS_SNPP_SP` — **science-quality** (standard processing). This is the right product for a
  historical / archival 12-month pull and is the default here.
- `VIIRS_SNPP_NRT` — near-real-time. Only needed for roughly the last ~2 months; the live site
  already uses NRT for its "last 3 days" fire layer.

The notebook pulls SP for the full window and automatically tops up the **most recent ~60 days**
with NRT, so the tail of the window isn't missed while SP catches up.

## 1. Parameters — change these if needed

In [ ]:
# --- Matching parameters ---
MATCH_RADIUS_M = 500        # a VIIRS detection within this many metres of a flare counts as that flare's
WINDOW_DAYS    = 365        # how far back to look for detections

# --- Iraq bounding box (west, south, east, north) ---
IRAQ_BBOX = "38.7,29.0,48.6,37.4"

# --- FIRMS sources to pull ---
SP_SOURCE  = "VIIRS_SNPP_SP"    # science-quality, used for the whole window
NRT_SOURCE = "VIIRS_SNPP_NRT"   # near-real-time, used to top up the most recent days
NRT_TOPUP_DAYS = 60             # how far back NRT typically needs to cover (SP lags ~1-2 months)

# --- Source of the flare list (the cached Overpass result committed to the repo) ---
FLARES_URL = 'https://raw.githubusercontent.com/rawazrauf/iraq-flaring-watch/main/query'

# --- Output file name (commit this to the repo root) ---
OUTPUT_FILE = 'flare_viirs_status.json'

# --- API behaviour ---
MAX_DAYS_PER_REQUEST = 5    # FIRMS Area API hard limit: at most 5 days per request
REQUEST_PAUSE_S      = 0.3  # small pause between requests to stay polite
RATE_LIMIT_TXNS      = 5000 # FIRMS allows 5000 transactions per 10-minute window

## 2. Your FIRMS MAP_KEY

Paste your key when asked on the first run. It is stored as a Colab Secret (`FIRMS_MAP_KEY`),
so subsequent runs pick it up automatically and never prompt again.

Get a free key at https://firms.modaps.eosdis.nasa.gov/api/map_key/ if you don't have one.

In [ ]:
# --- Your FIRMS MAP_KEY ---
# Best way (no typing each run): click the key icon (🔑) in Colab's LEFT sidebar,
# add a secret named exactly  FIRMS_MAP_KEY  with your key as the value, toggle
# "Notebook access" ON, then run this cell. It will be picked up automatically.
#
# If you have not set the secret, this cell will show a hidden prompt box at the
# BOTTOM of the cell output -- click into that box, paste your key, press Enter.
# Do NOT paste your key into this code cell.

MAP_KEY = None

# 1) Try the Colab Secret first.
try:
    from google.colab import userdata
    try:
        MAP_KEY = userdata.get('FIRMS_MAP_KEY')
    except Exception:
        MAP_KEY = None
except Exception:
    pass

# 2) Fall back to a hidden prompt (reads your input as TEXT, never as code).
if not MAP_KEY:
    import getpass
    MAP_KEY = getpass.getpass('Paste your FIRMS MAP_KEY here, then press Enter (input is hidden): ')

# Clean up stray spaces or quotation marks that sometimes come with a paste.
MAP_KEY = str(MAP_KEY).strip().strip('\'"').strip()

assert MAP_KEY, 'No MAP_KEY provided. Set the FIRMS_MAP_KEY secret or paste into the prompt box.'
assert ' ' not in MAP_KEY, 'MAP_KEY contains a space -- re-paste it cleanly.'
print('MAP_KEY loaded: %d characters, starts with %r.' % (len(MAP_KEY), MAP_KEY[:1]))

## 3. Verify the key and check your transaction budget

This hits the FIRMS `/mapkey_status` endpoint so we know the key works and how many
transactions you have left before we start pulling data.

In [ ]:
import urllib.request, urllib.error, json, time, math, csv, io, datetime

def _get(url, timeout=60):
    req = urllib.request.Request(url, headers={'User-Agent': 'iraq-flaring-watch/1.0'})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read().decode('utf-8', errors='replace')

status_url = f'https://firms.modaps.eosdis.nasa.gov/mapserver/mapkey_status/?MAP_KEY={MAP_KEY}'
try:
    s = _get(status_url)
    print(s)
except urllib.error.HTTPError as e:
    print('Could not read key status (continuing anyway):', e)

## 4. Load the flare list from GitHub
Reads the same `query` file the website uses, so flare IDs match exactly.

In [ ]:
with urllib.request.urlopen(FLARES_URL) as r:
    overpass = json.load(r)

flares = []
for el in overpass.get('elements', []):
    lat = el.get('lat') or (el.get('center') or {}).get('lat')
    lon = el.get('lon') or (el.get('center') or {}).get('lon')
    if lat is None or lon is None:
        continue
    flares.append({'id': el['id'], 'lat': float(lat), 'lon': float(lon)})

print(f'Loaded {len(flares)} flares from the repo query file.')
print('Example:', flares[0] if flares else 'none')

## 5. Pull VIIRS detections from the FIRMS Area API

The Area API returns at most **5 days per request**, so we step through the window in
5-day chunks. We pull **SP** (science-quality) for the whole window, then top up the most
recent ~60 days with **NRT** so nothing is missed while SP catches up.

A detection is kept only if it falls inside the Iraq bbox (the API already filters by bbox,
but we keep the parse defensive). Each row we keep is `(lat, lon, acq_date)`.

If you hit the 5000-transaction / 10-minute limit mid-run, the loop pauses and resumes
automatically.

In [ ]:
def firms_area_url(source, start_date, day_range):
    # /api/area/csv/MAP_KEY/SOURCE/AREA/DAY_RANGE/DATE
    return (f'https://firms.modaps.eosdis.nasa.gov/api/area/csv/'
            f'{MAP_KEY}/{source}/{IRAQ_BBOX}/{day_range}/{start_date.isoformat()}')

def fetch_window(source, overall_start, overall_end):
    """Fetch all detections for `source` from overall_start..overall_end (inclusive),
    stepping in <=5-day chunks. Returns a list of (lat, lon, date)."""
    out = []
    cur = overall_start
    chunk_no = 0
    while cur <= overall_end:
        days = min(MAX_DAYS_PER_REQUEST, (overall_end - cur).days + 1)
        url = firms_area_url(source, cur, days)
        chunk_no += 1
        for attempt in range(4):
            try:
                text = _get(url)
                break
            except urllib.error.HTTPError as e:
                if e.code in (429, 503):
                    wait = 60 * (attempt + 1)
                    print(f'  rate-limited ({e.code}); pausing {wait}s...')
                    time.sleep(wait)
                    continue
                raise
            except Exception as e:
                print(f'  retry after error: {e}')
                time.sleep(10)
        else:
            print(f'  giving up on chunk starting {cur}')
            cur += datetime.timedelta(days=days)
            continue

        # The first line that looks like an error message, not CSV, is worth surfacing.
        stripped = text.strip()
        if stripped and not stripped.lower().startswith('country_id') and 'latitude' not in stripped.split('\n',1)[0].lower():
            if 'Invalid' in stripped or 'error' in stripped.lower() or 'exceed' in stripped.lower():
                print(f'  API said: {stripped[:200]}')

        reader = csv.DictReader(io.StringIO(text))
        kept = 0
        for row in reader:
            try:
                lat = float(row['latitude']); lon = float(row['longitude'])
                d = datetime.date.fromisoformat(row['acq_date'].strip())
            except (KeyError, ValueError, TypeError):
                continue
            out.append((lat, lon, d))
            kept += 1
        if chunk_no % 10 == 0:
            print(f'  ...{chunk_no} chunks, {len(out)} detections so far (last chunk +{kept})')
        cur += datetime.timedelta(days=days)
        time.sleep(REQUEST_PAUSE_S)
    return out

today      = datetime.date.today()
win_start  = today - datetime.timedelta(days=WINDOW_DAYS)
nrt_start  = today - datetime.timedelta(days=NRT_TOPUP_DAYS)

print(f'Window: {win_start} -> {today}  ({WINDOW_DAYS} days)')
print(f'Pulling SP ({SP_SOURCE}) across the full window...')
sp_dets = fetch_window(SP_SOURCE, win_start, today)
print(f'  SP: {len(sp_dets)} detections')

print(f'Topping up the last {NRT_TOPUP_DAYS} days with NRT ({NRT_SOURCE})...')
nrt_dets = fetch_window(NRT_SOURCE, nrt_start, today)
print(f'  NRT: {len(nrt_dets)} detections')

# Merge, de-duplicate exact (lat,lon,date) rows that appear in both products.
detections = list({(round(la,5), round(lo,5), d) for (la,lo,d) in (sp_dets + nrt_dets)})
print(f'Merged + de-duplicated: {len(detections)} unique detections in the window.')

## 6. Match each flare to nearby detections (within 500 m)
Uses a simple spatial grid so it stays fast even with tens of thousands of detections.

In [ ]:
def haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlmb = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dlmb/2)**2
    return 2*R*math.asin(math.sqrt(a))

# Bin detections into ~0.01-degree grid cells (~1.1 km) so each flare only compares against nearby ones.
CELL = 0.01
grid = {}
for (lat, lon, d) in detections:
    key = (round(lat/CELL), round(lon/CELL))
    grid.setdefault(key, []).append((lat, lon, d))

status = {}
for f in flares:
    ck, cl = round(f['lat']/CELL), round(f['lon']/CELL)
    last_date = None
    count = 0
    for dk in (ck-1, ck, ck+1):
        for dl in (cl-1, cl, cl+1):
            for (lat, lon, d) in grid.get((dk, dl), []):
                if haversine_m(f['lat'], f['lon'], lat, lon) <= MATCH_RADIUS_M:
                    count += 1
                    if last_date is None or d > last_date:
                        last_date = d
    status[str(f['id'])] = {
        'last_detection': last_date.isoformat() if last_date else None,
        'detections': count
    }

matched = sum(1 for v in status.values() if v['last_detection'])
print(f'{matched} of {len(flares)} flares had at least one VIIRS detection within {MATCH_RADIUS_M} m in the window.')
print(f'{len(flares)-matched} flares had NO detection in the window.')

## 7. Write the output JSON + a little metadata, then download

In [ ]:
output = {
    '_meta': {
        'generated': datetime.date.today().isoformat(),
        'window_days': WINDOW_DAYS,
        'match_radius_m': MATCH_RADIUS_M,
        'source': 'NASA FIRMS VIIRS S-NPP (SP + NRT topup)',
        'bbox': IRAQ_BBOX,
        'flares': len(flares),
        'detections_in_window': len(detections)
    },
    'flares': status
}

with open(OUTPUT_FILE, 'w') as fh:
    json.dump(output, fh, ensure_ascii=False, separators=(',', ':'))

print('Wrote', OUTPUT_FILE)
try:
    from google.colab import files as _f
    _f.download(OUTPUT_FILE)   # commit this file to the repo root on GitHub
except Exception:
    print('(Not on Colab — find', OUTPUT_FILE, 'in the working directory.)')

## 8. (Optional) Quick sanity peek
Shows a few flares that have gone dark the longest in the window — a useful gut-check before
you commit the file.

In [ ]:
rows = []
for fid, v in status.items():
    rows.append((v['last_detection'] or '0000-00-00', v['detections'], fid))
rows.sort()

print('Flares with NO detection in the window (first 20):')
none_rows = [r for r in rows if r[0] == '0000-00-00']
for _, _, fid in none_rows[:20]:
    print('  ', fid)
print(f'  ...{len(none_rows)} total with no detection.\n')

print('Most recently seen (last 5):')
for last, cnt, fid in rows[-5:]:
    print(f'   {fid}: last {last}, {cnt} detections')